In [1]:
import glob
import pandas as pd

files = sorted(glob.glob("../data/raw/DAT_ASCII_EURUSD_M1_*.csv"))
cols = ["ts", "open", "high", "low", "close", "volume"]

df = pd.concat(
    (pd.read_csv(f, sep=";", header=None, names=cols) for f in files),
    ignore_index=True,
)
print(len(files), "files,", len(df), "rows")
df.head()

11 files, 4043464 rows


,ts,open,high,low,close,volume
0,20150101 130000,1.20965,1.20977,1.20962,1.20962,0
1,20150101 130100,1.20963,1.20968,1.20962,1.20962,0
2,20150101 130200,1.20965,1.20970,1.20961,1.20961,0
3,20150101 130400,1.20959,1.21008,1.20959,1.20983,0
4,20150101 130500,1.20988,1.20988,1.20988,1.20988,0


In [2]:
df["ts_est"] = pd.to_datetime(df["ts"], format="%Y%m%d %H%M%S")
df["ts_utc"] = (df["ts_est"] + pd.Timedelta(hours=5)).dt.tz_localize("UTC")
df["pair"] = "EURUSD"
df = df[["pair", "ts_utc", "open", "high", "low", "close", "volume"]]
df[["ts_utc"]].agg(["min", "max"])

,ts_utc
min,2015-01-01 18:00:00+00:00
max,2025-12-31 21:57:00+00:00


In [3]:
print("duplicates:", df.duplicated(["pair", "ts_utc"]).sum())
print("nulls:\n", df.isna().sum())
print("bad bars (high < low, or close outside range):",
      ((df.high < df.low) | (df.close > df.high) | (df.close < df.low)).sum())
print("rows per year:\n", df.groupby(df.ts_utc.dt.year).size())

duplicates: 360
nulls:
 pair      0
ts_utc    0
open      0
high      0
low       0
close     0
volume    0
dtype: int64
bad bars (high < low, or close outside range): 0
rows per year:
 ts_utc
2015    372210
2016    372679
2017    371635
2018    372607
2019    372530
2020    372335
2021    369502
2022    372865
2023    322638
2024    372379
2025    372084
dtype: int64


In [4]:
dups = df[df.duplicated(["pair", "ts_utc"], keep=False)].sort_values("ts_utc")
print("exact copies (every column the same):", df.duplicated().sum())
print(dups.groupby(dups.ts_utc.dt.date).size().head(10))
dups.head(6)

exact copies (every column the same): 360
ts_utc
2019-10-28    120
2020-10-26    120
2021-11-01    120
2022-10-31    120
2023-10-30    120
2025-10-27    120
dtype: int64


,pair,ts_utc,open,high,low,close,volume
1795089,EURUSD,2019-10-28 00:00:00+00:00,1.10839,1.10844,1.10836,1.10836,0
1795149,EURUSD,2019-10-28 00:00:00+00:00,1.10839,1.10844,1.10836,1.10836,0
1795150,EURUSD,2019-10-28 00:01:00+00:00,1.10837,1.10837,1.10826,1.10826,0
1795090,EURUSD,2019-10-28 00:01:00+00:00,1.10837,1.10837,1.10826,1.10826,0
1795091,EURUSD,2019-10-28 00:02:00+00:00,1.10826,1.10826,1.10822,1.10822,0
1795151,EURUSD,2019-10-28 00:02:00+00:00,1.10826,1.10826,1.10822,1.10822,0


In [5]:
df = df.drop_duplicates(["pair", "ts_utc"], keep="first").reset_index(drop=True)
print(len(df), "rows after dedupe")

4043104 rows after dedupe


In [6]:
import io
from sqlalchemy import create_engine, text

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS fx_m1;
        CREATE TABLE fx_m1 (
            pair    text        NOT NULL,
            ts_utc  timestamptz NOT NULL,
            open    double precision,
            high    double precision,
            low     double precision,
            close   double precision,
            volume  double precision,
            PRIMARY KEY (pair, ts_utc)
        );
    """))

buf = io.StringIO()
df.to_csv(buf, index=False, header=False)
buf.seek(0)

raw = engine.raw_connection()
try:
    with raw.cursor() as cur:
        cur.copy_expert("COPY fx_m1 FROM STDIN WITH (FORMAT csv)", buf)
    raw.commit()
finally:
    raw.close()

pd.read_sql("SELECT pair, COUNT(*) AS n, MIN(ts_utc), MAX(ts_utc) FROM fx_m1 GROUP BY pair", engine)

,pair,n,min,max
0,EURUSD,4043104,2015-01-01 18:00:00+00:00,2025-12-31 21:57:00+00:00


In [7]:
q = """
SELECT date_trunc('month', ts_utc) AS month,
       COUNT(*) AS bars
FROM fx_m1
WHERE pair = 'EURUSD'
GROUP BY 1
ORDER BY 1
"""
m = pd.read_sql(q, engine)
m["weekdays"] = m["month"].apply(
    lambda d: len(pd.bdate_range(d, d + pd.offsets.MonthEnd(0)))
)
m["coverage"] = (m["bars"] / (m["weekdays"] * 1440)).round(3)
print(m["coverage"].describe())
m.sort_values("coverage").head(10)

count    132.000000
mean       0.978470
std        0.057875
min        0.691000
25%        0.983750
50%        0.995000
75%        0.999000
max        1.012000
Name: coverage, dtype: float64


,month,bars,weekdays,coverage
101,2023-06-01 04:00:00+00:00,21893,22,0.691
98,2023-03-01 04:00:00+00:00,22966,23,0.693
99,2023-04-01 04:00:00+00:00,20113,20,0.698
100,2023-05-01 04:00:00+00:00,23354,23,0.705
102,2023-07-01 04:00:00+00:00,22044,21,0.729
97,2023-02-01 04:00:00+00:00,25670,20,0.891
35,2017-12-01 04:00:00+00:00,28350,21,0.938
11,2015-12-01 04:00:00+00:00,31088,23,0.939
131,2025-12-01 04:00:00+00:00,31407,23,0.948
107,2023-12-01 04:00:00+00:00,28691,21,0.949


In [8]:
q = """
SELECT EXTRACT(hour FROM ts_utc AT TIME ZONE 'UTC') AS hour_utc,
       COUNT(*) FILTER (WHERE ts_utc >= '2023-03-01' AND ts_utc < '2023-08-01') AS gap_period,
       COUNT(*) FILTER (WHERE ts_utc >= '2022-03-01' AND ts_utc < '2022-08-01') AS same_months_2022
FROM fx_m1
GROUP BY 1
ORDER BY 1
"""
h = pd.read_sql(q, engine)
h["ratio"] = (h["gap_period"] / h["same_months_2022"]).round(2)
h

,hour_utc,gap_period,same_months_2022,ratio
0,0.0,3882,6470,0.60
1,1.0,6480,6540,0.99
2,2.0,6480,6540,0.99
3,3.0,6479,6539,0.99
4,4.0,6476,6536,0.99
5,5.0,6475,6538,0.99
6,6.0,6478,6539,0.99
7,7.0,6480,6540,0.99
8,8.0,6480,6540,0.99
9,9.0,6420,6540,0.98


In [9]:
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE fx_m1 ADD COLUMN IF NOT EXISTS bad_period boolean DEFAULT false"))
    conn.execute(text("""
        UPDATE fx_m1 SET bad_period = true
        WHERE pair = 'EURUSD'
          AND ts_utc >= '2023-03-01' AND ts_utc < '2023-08-01'
    """))

pd.read_sql("SELECT bad_period, COUNT(*) FROM fx_m1 GROUP BY 1", engine)

,bad_period,count
0,False,3932734
1,True,110370
